# HCDE 530 — Week 5 In-Class Activity
### Five Questions — app_reviews_demo.csv

Answer each question using pandas. A plain-English comment above each code block explains what question it answers.

In [9]:
import pandas as pd

df = pd.read_csv('app_reviews_demo.csv')

---
## Question 1 — What does your dataset look like?

In [10]:
# What do the first rows look like? Are there any obvious issues?
df.head()

,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3


In [11]:
df.tail()

,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
495,496,Dovetail,research repository,5,Bulk-tagging across sessions is a huge time sa...,2023-10-02,46,True,desktop,1.8.4
496,497,Maze,usability testing,3,The fundamentals are strong; the extras feel h...,2024-02-14,29,True,desktop,4.2.3
497,498,Miro,collaborative whiteboard,5,Video clips export cleanly with timecodes intact.,2023-06-20,7,True,tablet,9.3.0
498,499,Fieldkit,field research,5,Custom fields on participants let me filter by...,2024-08-05,43,True,mobile,3.0.0
499,500,Lookback,user research,4,The guest access feature is perfect for extern...,2024-05-09,31,True,mobile,5.1.2


In [12]:
# What columns do we have, what type is each one, and how many non-null values are there?
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


The dataset has 500 rows and 10 columns. Most columns are complete, but two have missing values: `device_type` (63 missing) and `app_version` (111 missing). The columns cover the app name, category, a 1–5 star rating, the review text, a date, helpful vote count, a verified purchase flag, the device type, and the app version. The mix of numeric, text, boolean, and categorical columns means different pandas operations will apply to different columns.

---
## Question 2 — What's the distribution of your most important column?

The most important column here is `rating` — it's the core signal in any app review dataset.

In [13]:
# How are ratings distributed? Is the data balanced or skewed toward high/low scores?
counts = df['rating'].value_counts().sort_index()
pcts   = df['rating'].value_counts(normalize=True).sort_index().mul(100).round(1)
pd.concat([counts, pcts], axis=1, keys=['count', '%'])

,count,%
rating,,
1,29,5.8
2,43,8.6
3,61,12.2
4,160,32.0
5,207,41.4


Ratings are heavily skewed toward the positive end. 5-star reviews are the single largest group (207, 41.4%) and 4-star reviews are the second largest (160, 32%), meaning nearly three quarters of all reviews are positive. Negative reviews (1 and 2 stars) make up only about 14% combined. This kind of positivity skew is common in app review datasets and is worth keeping in mind — averages will be pulled upward, so a "low" average of 3.67 actually represents a meaningful drop from the norm.

---
## Question 3 — Filter to a meaningful subset. What's in it?

In [14]:
# Which reviews are negative (rating below 3), and how much of the dataset are they?
negative = df[df['rating'] < 3]

print(f"{len(negative)} negative reviews ({len(negative)/len(df):.1%} of total)")
negative.head(10)

72 negative reviews (14.4% of total)


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
26,27,Fieldkit,field research,2,No built-in way to generate a structured debri...,2024-04-04,8,True,mobile,2.5.0
31,32,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-06-27,10,True,tablet,4.2.3
32,33,Lookback,user research,2,Loading large projects takes noticeably longer...,2024-11-22,15,True,desktop,5.2.0
42,43,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-19,9,True,mobile,NaN
51,52,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-09-12,32,True,mobile,NaN
74,75,Miro,collaborative whiteboard,1,I've lost tags twice after a session due to a ...,2023-12-25,14,True,desktop,NaN
79,80,Dovetail,research repository,2,The Figma integration is read-only; I can't pu...,2024-08-02,2,True,desktop,1.8.4
102,103,Miro,collaborative whiteboard,2,Storage limits hit quickly when you're recordi...,2024-09-10,16,True,desktop,NaN


There are 72 negative reviews (ratings of 1 or 2), which is 14.4% of the full dataset. Fieldkit appears frequently in this subset — consistent with it having the lowest average rating overall (3.67). The negative reviews tend to mention specific friction points like slow search, missing features, and transcription accuracy issues. Even though this is a small slice of the data, it is often the most actionable for product teams because it surfaces concrete problems rather than general satisfaction.

---
## Question 4 — Group by a category and find the average of a numeric column.

In [15]:
# Which app has the highest average rating, and how many reviews does each app have?
df.groupby('app')['rating'].agg(['mean', 'count']).round(2).sort_values('mean')

,mean,count
app,,
Fieldkit,3.67,92
Lookback,3.90,105
Maze,4.00,93
Miro,4.02,121
Dovetail,4.12,89


Dovetail has the highest average rating (4.12) and Fieldkit has the lowest (3.67), a gap of 0.45 stars. The five apps have similar review counts (89–121), so the differences in means are not simply due to one app having far fewer reviews. Miro has the most reviews (121) and sits near the middle (4.02). The spread between apps is relatively narrow — all means fall between 3.67 and 4.12 — but whether those differences are statistically meaningful requires a significance test.

---
## Question 5 — Where are the missing values? Are any columns incomplete?

In [16]:
# How many values are missing per column, and what percentage of the column is that?
missing_count = df.isnull().sum()
missing_pct   = df.isnull().mean().mul(100).round(1)
summary = pd.concat([missing_count, missing_pct], axis=1, keys=['missing', '%'])
summary[summary['missing'] > 0]

,missing,%
device_type,63,12.6
app_version,111,22.2


Two columns have missing data: `device_type` is missing 63 values (12.6% of rows) and `app_version` is missing 111 values (22.2%). Every other column is complete. The `app_version` gap is significant — more than one in five rows — so any analysis that groups or filters by version should account for this. For `device_type`, 63 missing rows is manageable but not trivial; dropping them silently would remove 12.6% of the data, which could skew results if the missing rows are not random.